# 00 — Setup and data checks
Read-only. Verifies that the dataset package, split table, gene order, PPI edges and availability table are present and mutually consistent. Writes nothing.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, pickle, pandas as pd
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import read_genes, build_ppi_edge_index, ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability

REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)             # asserts SHA-256
print('dataset hash OK;', len(pyg_dataset), 'slides;', len(gene_order), 'genes')
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
print('split train/val/test/HD =', len(train_s), len(val_s), len(test_s), len(hd_s))

# gene order table == pkg gene_order
assert read_genes(T / 'sasp54_genes_canonical_order.csv') == gene_order
# PPI edges: CSV construction must equal the construction from the networkx graph in the package
e_csv = build_ppi_edge_index(gene_order, T / 'ppi_string_v12_edges.csv')
e_nx  = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order)
import torch
assert torch.equal(e_csv, e_nx) and e_csv.shape == (2, 2 * N_PPI_UNDIRECTED_EDGES)
print('PPI edge_index (CSV) == PPI edge_index (package graph):', tuple(e_csv.shape))
# split table == split recorded in the package
sp = pd.read_csv(T / 'slide_partition.csv').set_index('slide_id').split_assignment
assert all(sp[s] == d.split for s, d in pyg_dataset.items())
print('slide_partition.csv matches package splits for all', len(sp), 'slides')
# availability table == residual_zero_fill_log on Drive
a_repo = pd.read_csv(T / 'availability_unmeasured.csv')
a_drv  = pd.read_csv(Path(BASE) / 'checkpoint_2026-09-25' / 'residual_zero_fill_log_2026-09-26.csv')
assert a_repo[['sample_id','gene']].values.tolist() == a_drv[['sample_id','gene']].values.tolist()
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
print('unmeasured slide-gene pairs:', LOGGED)
print('ALL SETUP CHECKS PASSED')